# M6 scam detection: XLM-RoBERTa on Colab

Fine-tunes **xlm-roberta-base** with the **same 5 folds** as the local models A/B/C
(`fold` column of `m6_dataset.csv`), so the scores are directly comparable.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.
Then run the cells from top to bottom (**Runtime → Run all**). Step 2 asks you to upload `m6_dataset.csv`.
Total time on a T4: about 10 minutes.

## 1. Check the GPU and install libraries

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q transformers accelerate scikit-learn
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

## 2. Upload the dataset
Pick `ml/m6/data/m6_dataset.csv` from the project folder on your laptop.

In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
df = pd.read_csv(name, keep_default_na=False, dtype={"is_scam": int, "fold": int})
print(f"{len(df)} rows, {df['group'].nunique()} groups, folds {sorted(df['fold'].unique().tolist())}")
print(df["is_scam"].value_counts().rename({0: "legit", 1: "scam"}).to_dict())

## 3. Text cleaner (same code as `ml/m6/textnorm.py`)

In [ ]:
import re

TATWEEL = "ـ"
_LEET = [
    (re.compile(r"(?<=[a-z])@|@(?=[a-z])"), "a"),
    (re.compile(r"(?<=[a-z])€|€(?=[a-z])"), "e"),
    (re.compile(r"(?<=[a-z])!(?=[a-z])"), "i"),
    (re.compile(r"(?<![0-9])(?<=[a-z])0|0(?=[a-z])(?![0-9])"), "o"),
    (re.compile(r"(?<=[a-z])1(?=[a-z])"), "i"),
    (re.compile(r"(?<=[0-9])o\b"), "0"),  # "8O dt" -> "80 dt" (letter O used as zero)
]
# Three or more single characters split by a space, dot, underscore or hyphen: "f l o u s", "t.e.l.e.g.r.a.m".
_SPACED = re.compile(r"(?<!\w)(?:\w[ ._-]){2,}\w(?!\w)")


def _join(match: re.Match) -> str:
    return re.sub(r"[ ._-]", "", match.group(0))


def normalize(text: str) -> str:
    t = text.replace(TATWEEL, "").lower()
    for pattern, repl in _LEET:
        t = pattern.sub(repl, t)
    t = _SPACED.sub(_join, t)
    t = re.sub(r"(?<=[^\W_])_(?=[^\W_])", "", t)  # "kh_a_m_s_i_n" -> "khamsin"
    return re.sub(r"\s+", " ", t).strip()


df["clean"] = df["text"].map(normalize)
df[["text", "clean"]].sample(5, random_state=1)

## 4. Settings
Fixed hyper-parameters, chosen in advance (not tuned on these folds), so the comparison with the
local models stays fair.

In [ ]:
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 128
EPOCHS = 5
BATCH = 16
LR = 2e-5
WARMUP = 0.1
SEED = 42

## 5. Train and predict, one fold at a time
Each fold is predicted by a model that never saw its groups (out-of-fold).

In [ ]:
import numpy as np, random, time
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
device = torch.device("cuda")

def encode(texts):
    e = tok(list(texts), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    return e["input_ids"], e["attention_mask"]

oof = np.zeros(len(df))
start = time.time()
for k in sorted(df["fold"].unique()):
    set_seed(SEED)
    tr, te = df[df["fold"] != k], df[df["fold"] == k]
    ids, mask = encode(tr["clean"])
    y = torch.tensor(tr["is_scam"].to_numpy())
    loader = DataLoader(TensorDataset(ids, mask, y), batch_size=BATCH, shuffle=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
    # Balanced class weights, like class_weight="balanced" in the local models.
    counts = np.bincount(tr["is_scam"], minlength=2)
    weights = torch.tensor(len(tr) / (2 * counts), dtype=torch.float, device=device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    scaler = torch.amp.GradScaler("cuda")

    model.train()
    for epoch in range(EPOCHS):
        total = 0.0
        for b_ids, b_mask, b_y in loader:
            b_ids, b_mask, b_y = b_ids.to(device), b_mask.to(device), b_y.to(device)
            opt.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(model(input_ids=b_ids, attention_mask=b_mask).logits, b_y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            total += loss.item()
        print(f"fold {k} epoch {epoch + 1}/{EPOCHS}  loss {total / len(loader):.4f}")

    model.eval()
    ids, mask = encode(te["clean"])
    probs = []
    with torch.no_grad():
        for i in range(0, len(ids), 64):
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(input_ids=ids[i:i+64].to(device), attention_mask=mask[i:i+64].to(device)).logits
            probs.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    oof[te.index.to_numpy()] = np.concatenate(probs)
    del model, opt; torch.cuda.empty_cache()
    print(f"fold {k} done ({time.time() - start:.0f}s total)\n")

df["xlmr_prob"] = oof

## 6. Results (same measures as the local report)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

THRESHOLD = 0.5
y = df["is_scam"].to_numpy()
pred = (oof >= THRESHOLD).astype(int)
scam, legit = df["is_scam"] == 1, df["is_scam"] == 0
hard_neg = legit & (df["difficulty"] == "hard_negative")
reward = legit & df["template_id"].str.contains("REWARD|PAID-AFTER|RIB-TO-PAY")
obf = scam & (df["difficulty"] == "obfuscated")

def rate(m):
    return f"{int(pred[m.to_numpy()].sum())}/{int(m.sum())}"

fold_f1 = [f1_score(y[df["fold"] == k], pred[df["fold"] == k]) for k in sorted(df["fold"].unique())]
print(f"XLM-R  precision {precision_score(y, pred):.2f}  recall {recall_score(y, pred):.2f}  "
      f"F1 {f1_score(y, pred):.2f}  (folds {min(fold_f1):.2f}-{max(fold_f1):.2f})  PR-AUC {average_precision_score(y, oof):.2f}")
print(f"scams caught {rate(scam)} | legit flagged {rate(legit)} | hard-neg flagged {rate(hard_neg)} | "
      f"reward posts flagged {rate(reward)} | obfuscated caught {rate(obf)}")
print()
for lang in sorted(df["language"].unique()):
    m = df["language"] == lang
    print(f"  {lang:14} scams caught {rate(m & scam):>7}   legit flagged {rate(m & legit):>7}")

## 7. Download the predictions
Send the downloaded `m6_xlmr_oof.csv` back so it can be compared row by row with model C.

In [ ]:
out = df[["id", "fold", "is_scam", "xlmr_prob"]]
out.to_csv("m6_xlmr_oof.csv", index=False)
files.download("m6_xlmr_oof.csv")